# Preprocess radiance-only data
A few datasets were converted directly to radiance on-board the UAV. These do not fit the regular pipeline processing and need to be manually converted to 1a_radiance + imudata datasets.

The datasets are from Juvika in April 2021 and from Sandsund in April 2021 / May 2021.

In [1]:
import massipipe.pipeline
from massipipe.georeferencing import ImuDataParser
from massipipe.irradiance import IrradianceConverter
from massipipe.utils import resample_cube_spectrally, read_envi, array_to_header_string, save_envi
from massipipe.reflectance import ReflectanceConverter
from pathlib import Path
import shutil
from rich import print
import matplotlib.pyplot as plt

In [2]:
base_dir = Path('/home/mha114/data/seabee-minio/')
example_image_300_bands = Path('/home/mha114/data/seabee-minio/bodo/juvika/aerial/hsi/20220624/massimal_bodo_juvika_202206241528_hsi/0_raw/Juvika_June2022-1/Juvika_June2022_Pika_L_1.bil.hdr')

In [3]:
radiance_dirs = sorted(d for d in base_dir.rglob("**/0_radiance"))
data_dirs = [d.parent for d in radiance_dirs]

In [4]:
for data_dir in data_dirs:
    print(data_dir)
    # print([f.name for f in sorted(data_dir.glob('*'))])
    
#     raw_dir = data_dir / "0_raw"
#     if raw_dir.exists():
#         print(f"Removing {raw_dir}")  
#         shutil.rmtree(raw_dir)
    
#     imudata_dir = data_dir / "imudata"
#     if imudata_dir.exists():
#         print(f"Removing {imudata_dir}")   
#         shutil.rmtree(imudata_dir)
    
#     proc_rad_dir = data_dir / "1a_radiance"
#     if proc_rad_dir.exists():
#         print(f"Removing {proc_rad_dir}")   
#         shutil.rmtree(proc_rad_dir)


/home/mha114/data/seabee-minio/bodo/juvika/aerial/hsi/20210418/massimal_bodo_juvika_202104181128-transects-07m_hsi

/home/mha114/data/seabee-minio/bodo/juvika/aerial/hsi/20210418/massimal_bodo_juvika_202104181131-transects-15m_hsi

/home/mha114/data/seabee-minio/bodo/juvika/aerial/hsi/20210418/massimal_bodo_juvika_202104181135-transects-30m_hsi

/home/mha114/data/seabee-minio/bodo/juvika/aerial/hsi/20210418/massimal_bodo_juvika_202104181158_hsi

/home/mha114/data/seabee-minio/bodo/sandsund/aerial/hsi/20210418/massimal_bodo_sandsund_202104181440_hsi

/home/mha114/data/seabee-minio/bodo/sandsund/aerial/hsi/20210520/massimal_bodo_sandsund_202105201139-area_hsi

/home/mha114/data/seabee-minio/bodo/sandsund/aerial/hsi/20210520/massimal_bodo_sandsund_202105201233-transects_hsi

In [5]:
def raw_rad_image_number(raw_rad_image_path):
    raw_rad_image_path = Path(raw_rad_image_path)
    image_file_stem = raw_rad_image_path.name.split("-radiance.")[0]
    image_number = image_file_stem.split("_")[-1]
    return int(image_number)

In [6]:
_, wl_300, _ = read_envi(example_image_300_bands)
wl_300_str = array_to_header_string(wl_300)
# print(wl_300_str)

In [7]:
imu_data_parser = ImuDataParser()
reflectance_converter = ReflectanceConverter()

In [8]:
for data_dir in data_dirs:
    print(f"**** {data_dir.name} ****")
    raw_rad_dir = data_dir / "0_radiance"
    raw_rad_image_paths = sorted(raw_rad_dir.rglob("*radiance*.bil.hdr"),
                          key=lambda x: raw_rad_image_number(x))
    base_names = [f"{data_dir.name}_{i:03d}" for i in range(len(raw_rad_image_paths))]
    
    # Create irradiance converter
    irradiance_converter = IrradianceConverter(irrad_cal_file=data_dir/"calibration"/"FLMS16638_Radiometric_Jan2021.dcp")
    
    # Create output directories
    rad_dir = data_dir / "1a_radiance"
    rad_dir.mkdir(exist_ok=True)
    imudata_dir = data_dir / "imudata"
    imudata_dir.mkdir(exist_ok=True)
    
    for raw_rad_im_path,base_name in zip(raw_rad_image_paths,base_names):
        print(f"Reading {raw_rad_im_path.name}")
        
        # Read and resample image
        raw_rad_im, raw_rad_wl, metadata = read_envi(raw_rad_im_path)
        rad_im = resample_cube_spectrally(raw_rad_im,raw_rad_wl,wl_300)
        
        # Save resampled image with standard name
        metadata["wavelength"] = wl_300_str
        rad_im_path = rad_dir / (base_name + "_radiance.bip.hdr")
        print(f"Saving processed radiance file to {rad_im_path.name}")
        save_envi(rad_im_path,rad_im,metadata)
        
        # Parse and save IMU data
        imu_base_name = raw_rad_im_path.name.split("-radiance.")[0]
        lcf_path = raw_rad_im_path.parent / (imu_base_name + ".lcf")
        times_path = raw_rad_im_path.parent / (imu_base_name + ".bil.times")
        imudata_path = imudata_dir / (base_name + "_imudata.json")
        print(f"Reading {lcf_path.name} and {times_path.name}")
        print(f"Saving IMU data to {imudata_path.name}")
        imu_data_parser.read_and_save_imu_data(lcf_path,times_path,imudata_path)
        
        # Calibrate and process irradiance data
        raw_spec_base_name = raw_rad_im_path.name.split("_Pika_L")[0]
        image_number = raw_rad_image_number(raw_rad_im_path)
        raw_spec_path = raw_rad_im_path.parent / (
            f"{raw_spec_base_name}_downwelling_{image_number}_pre.spec.hdr")
        irrad_path = rad_dir / (base_name + "_irradiance.spec.hdr")
        print(f"Reading raw spec {raw_spec_path.name}")
        print(f"Saving irradiance {irrad_path.name}")
        irradiance_converter.convert_raw_file_to_irradiance(raw_spec_path,irrad_path)
        
        # Add irradiance to radiance header
        reflectance_converter.add_irradiance_spectrum_to_header(rad_im_path,irrad_path)

**** massimal_bodo_juvika_202104181128-transects-07m_hsi ****

15:24:30 INFO: Non-empty downwelling calibration directory /home/mha114/data/seabee-minio/bodo/juvika/aerial/hsi/20210418/massimal_bodo_juvika_202104181128-transects-07m_hsi/calibration/downwelling_calibration_spectra
15:24:30 INFO: Skipping unzipping of downwelling calibration file, assuming unzipping already done.


Reading manual_Pika_L_1-radiance.bil.hdr

Saving processed radiance file to massimal_bodo_juvika_202104181128-transects-07m_hsi_000_radiance.bip.hdr

Reading manual_Pika_L_1.lcf and manual_Pika_L_1.bil.times

...

Saving processed radiance file to massimal_bodo_sandsund_202105201233-transects_hsi_007_radiance.bip.hdr

Reading manual_Pika_L_8.lcf and manual_Pika_L_8.bil.times

Saving IMU data to massimal_bodo_sandsund_202105201233-transects_hsi_007_imudata.json

Reading raw spec manual_downwelling_8_pre.spec.hdr

Saving irradiance massimal_bodo_sandsund_202105201233-transects_hsi_007_irradiance.spec.hdr

In [9]:
# plt.plot(raw_rad_wl,raw_rad_im[1000,450,:])
# plt.plot(wl_300,rad_im[1000,450,:])
# plt.show()